# Numerical Integration Methods Comparison

## Overview
Most neural ODEs have no analytical solution. We compare numerical methods: Forward Euler, Runge-Kutta 2 (RK2), and Runge-Kutta 4 (RK4).

## Mathematical Framework

Given an ODE:
$$\frac{dx}{dt} = f(x, t)$$

### Forward Euler (Order 1):
$$x_{n+1} = x_n + \Delta t \cdot f(x_n, t_n)$$

- **Pros**: Simple, fast
- **Cons**: Low accuracy, can be unstable
- **Error**: O(Δt)

### Runge-Kutta 2 (Midpoint Method, Order 2):
$$k_1 = f(x_n, t_n)$$
$$k_2 = f(x_n + \frac{\Delta t}{2}k_1, t_n + \frac{\Delta t}{2})$$
$$x_{n+1} = x_n + \Delta t \cdot k_2$$

- **Pros**: Good balance of speed/accuracy
- **Cons**: 2 function evaluations per step
- **Error**: O(Δt²)

### Runge-Kutta 4 (Order 4):
$$k_1 = f(x_n, t_n)$$
$$k_2 = f(x_n + \frac{\Delta t}{2}k_1, t_n + \frac{\Delta t}{2})$$
$$k_3 = f(x_n + \frac{\Delta t}{2}k_2, t_n + \frac{\Delta t}{2})$$
$$k_4 = f(x_n + \Delta t \cdot k_3, t_n + \Delta t)$$
$$x_{n+1} = x_n + \frac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

- **Pros**: Very accurate, stable
- **Cons**: 4 function evaluations per step
- **Error**: O(Δt⁴)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)

## Test Case: Leaky Integrate-and-Fire (LIF) Neuron

$$\tau_m \frac{dV}{dt} = -(V - V_{\text{rest}}) + RI$$

This model has an analytical solution for comparison.

In [ ]:
# LIF parameters
tau_m = 10.0      # Membrane time constant (ms)
V_rest = -65.0    # Resting potential (mV)
V_thresh = -50.0  # Spike threshold (mV)
V_reset = -70.0   # Reset potential (mV)
R = 10.0          # Membrane resistance (MΩ)
I_ext = 2.0       # External current (nA)

# ODE
def lif_ode(V, t, I):
    """LIF neuron ODE"""
    return (-(V - V_rest) + R * I) / tau_m

# Analytical solution (no spikes)
def lif_analytical(t, V0, I):
    """Analytical solution of LIF ODE"""
    V_inf = V_rest + R * I
    return V_inf + (V0 - V_inf) * np.exp(-t / tau_m)

print(f"LIF Neuron Parameters:")
print(f"  τₘ = {tau_m} ms")
print(f"  V_rest = {V_rest} mV")
print(f"  R = {R} MΩ")
print(f"  I = {I_ext} nA")
print(f"  Steady-state voltage: {V_rest + R * I_ext:.2f} mV")

## Implement Integration Methods

In [ ]:
def euler_forward(f, y0, t, args=()):
    """Forward Euler method"""
    n = len(t)
    y = np.zeros(n)
    y[0] = y0
    
    for i in range(1, n):
        dt = t[i] - t[i-1]
        y[i] = y[i-1] + dt * f(y[i-1], t[i-1], *args)
    
    return y

def rk2(f, y0, t, args=()):
    """Runge-Kutta 2 (Midpoint) method"""
    n = len(t)
    y = np.zeros(n)
    y[0] = y0
    
    for i in range(1, n):
        dt = t[i] - t[i-1]
        k1 = f(y[i-1], t[i-1], *args)
        k2 = f(y[i-1] + dt/2 * k1, t[i-1] + dt/2, *args)
        y[i] = y[i-1] + dt * k2
    
    return y

def rk4(f, y0, t, args=()):
    """Runge-Kutta 4 method"""
    n = len(t)
    y = np.zeros(n)
    y[0] = y0
    
    for i in range(1, n):
        dt = t[i] - t[i-1]
        k1 = f(y[i-1], t[i-1], *args)
        k2 = f(y[i-1] + dt/2 * k1, t[i-1] + dt/2, *args)
        k3 = f(y[i-1] + dt/2 * k2, t[i-1] + dt/2, *args)
        k4 = f(y[i-1] + dt * k3, t[i-1] + dt, *args)
        y[i] = y[i-1] + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
    
    return y

## Test with Different Time Steps

In [ ]:
# Simulation parameters
T = 50.0
V0 = -70.0

dt_values = [0.1, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, dt in enumerate(dt_values):
    t = np.arange(0, T, dt)
    
    # Analytical solution
    V_analytical = lif_analytical(t, V0, I_ext)
    
    # Numerical solutions
    start = time.time()
    V_euler = euler_forward(lif_ode, V0, t, args=(I_ext,))
    time_euler = time.time() - start
    
    start = time.time()
    V_rk2 = rk2(lif_ode, V0, t, args=(I_ext,))
    time_rk2 = time.time() - start
    
    start = time.time()
    V_rk4 = rk4(lif_ode, V0, t, args=(I_ext,))
    time_rk4 = time.time() - start
    
    # Plot
    ax = axes[idx]
    ax.plot(t, V_analytical, 'k--', linewidth=2, label='Analytical', alpha=0.8)
    ax.plot(t, V_euler, 'r-', linewidth=1.5, label='Euler', alpha=0.7)
    ax.plot(t, V_rk2, 'g-', linewidth=1.5, label='RK2', alpha=0.7)
    ax.plot(t, V_rk4, 'b-', linewidth=1.5, label='RK4', alpha=0.7)
    
    ax.set_xlabel('Time (ms)', fontsize=11)
    ax.set_ylabel('Voltage (mV)', fontsize=11)
    ax.set_title(f'Δt = {dt} ms', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Add timing info
    ax.text(0.98, 0.02, f'Euler: {time_euler*1000:.2f} ms\nRK2: {time_rk2*1000:.2f} ms\nRK4: {time_rk4*1000:.2f} ms',
            transform=ax.transAxes, fontsize=8, verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

fig.suptitle('Integration Methods Comparison: LIF Neuron', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/user/integration_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Error Analysis

In [ ]:
# Compute errors for different time steps
dt_range = np.logspace(-2, 0, 20)  # 0.01 to 1.0
errors_euler = []
errors_rk2 = []
errors_rk4 = []

for dt in dt_range:
    t = np.arange(0, T, dt)
    V_analytical = lif_analytical(t, V0, I_ext)
    
    V_euler = euler_forward(lif_ode, V0, t, args=(I_ext,))
    V_rk2 = rk2(lif_ode, V0, t, args=(I_ext,))
    V_rk4 = rk4(lif_ode, V0, t, args=(I_ext,))
    
    # Root mean square error
    errors_euler.append(np.sqrt(np.mean((V_euler - V_analytical)**2)))
    errors_rk2.append(np.sqrt(np.mean((V_rk2 - V_analytical)**2)))
    errors_rk4.append(np.sqrt(np.mean((V_rk4 - V_analytical)**2)))

# Plot error vs time step
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

ax.loglog(dt_range, errors_euler, 'ro-', linewidth=2, markersize=6, label='Euler (order 1)')
ax.loglog(dt_range, errors_rk2, 'go-', linewidth=2, markersize=6, label='RK2 (order 2)')
ax.loglog(dt_range, errors_rk4, 'bo-', linewidth=2, markersize=6, label='RK4 (order 4)')

# Reference lines for theoretical scaling
ax.loglog(dt_range, 0.5*dt_range, 'r--', alpha=0.5, linewidth=1.5, label='O(Δt)')
ax.loglog(dt_range, 0.05*dt_range**2, 'g--', alpha=0.5, linewidth=1.5, label='O(Δt²)')
ax.loglog(dt_range, 0.001*dt_range**4, 'b--', alpha=0.5, linewidth=1.5, label='O(Δt⁴)')

ax.set_xlabel('Time Step Δt (ms)', fontsize=12)
ax.set_ylabel('RMSE (mV)', fontsize=12)
ax.set_title('Error Scaling with Time Step', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('/home/user/integration_error_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nError scaling observations:")
print("  - Euler: error ∝ Δt (linear in log-log)")
print("  - RK2: error ∝ Δt² (slope of 2)")
print("  - RK4: error ∝ Δt⁴ (slope of 4)")

## Computational Efficiency Analysis

In [ ]:
# Trade-off: accuracy vs computational cost
target_errors = np.logspace(-4, -1, 20)

costs_euler = []
costs_rk2 = []
costs_rk4 = []

for target_error in target_errors:
    # Find dt that achieves target error
    
    # For Euler: error ∝ dt
    dt_euler = target_error / 0.5
    n_steps_euler = int(T / dt_euler)
    cost_euler = n_steps_euler * 1  # 1 function evaluation per step
    costs_euler.append(cost_euler)
    
    # For RK2: error ∝ dt²
    dt_rk2 = np.sqrt(target_error / 0.05)
    n_steps_rk2 = int(T / dt_rk2)
    cost_rk2 = n_steps_rk2 * 2  # 2 function evaluations per step
    costs_rk2.append(cost_rk2)
    
    # For RK4: error ∝ dt⁴
    dt_rk4 = (target_error / 0.001)**(1/4)
    n_steps_rk4 = int(T / dt_rk4)
    cost_rk4 = n_steps_rk4 * 4  # 4 function evaluations per step
    costs_rk4.append(cost_rk4)

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

ax.loglog(target_errors, costs_euler, 'ro-', linewidth=2, markersize=6, label='Euler')
ax.loglog(target_errors, costs_rk2, 'go-', linewidth=2, markersize=6, label='RK2')
ax.loglog(target_errors, costs_rk4, 'bo-', linewidth=2, markersize=6, label='RK4')

ax.set_xlabel('Target Error (mV)', fontsize=12)
ax.set_ylabel('Computational Cost (function evaluations)', fontsize=12)
ax.set_title('Accuracy vs Computational Cost Trade-off', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

# Invert x-axis (smaller error = higher accuracy)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('/home/user/integration_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEfficiency observations:")
print("  - For low accuracy: Euler is most efficient")
print("  - For high accuracy: RK4 is most efficient")
print("  - RK2 is a good compromise for moderate accuracy")

## Stability Analysis: Stiff Equations

In [ ]:
# Test with fast time constant (stiff system)
def stiff_ode(y, t):
    """Stiff ODE: dy/dt = -100*y"""
    return -100 * y

T = 1.0
y0 = 1.0

# Analytical
t_fine = np.linspace(0, T, 1000)
y_analytical = y0 * np.exp(-100 * t_fine)

# Test different dt
dt_values = [0.01, 0.02, 0.03, 0.05]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, dt in enumerate(dt_values):
    t = np.arange(0, T, dt)
    
    try:
        y_euler = euler_forward(stiff_ode, y0, t)
    except:
        y_euler = np.nan * np.ones_like(t)
    
    y_rk2 = rk2(stiff_ode, y0, t)
    y_rk4 = rk4(stiff_ode, y0, t)
    
    ax = axes[idx]
    ax.plot(t_fine, y_analytical, 'k--', linewidth=2, label='Analytical')
    ax.plot(t, y_euler, 'r-', linewidth=1.5, label='Euler', alpha=0.7)
    ax.plot(t, y_rk2, 'g-', linewidth=1.5, label='RK2', alpha=0.7)
    ax.plot(t, y_rk4, 'b-', linewidth=1.5, label='RK4', alpha=0.7)
    
    ax.set_xlabel('Time', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_title(f'Δt = {dt}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-2, 2])

fig.suptitle('Stability for Stiff Equation: dy/dt = -100y', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/user/integration_stability.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nStability observations:")
print("  - Euler becomes unstable for large dt")
print("  - RK4 is more stable but still has limits")
print("  - For stiff systems, use implicit methods (e.g., backward Euler)")

## Key Takeaways

1. **Trade-offs**: Accuracy vs computational cost vs stability
2. **Euler**: Fast but inaccurate; use for quick prototyping
3. **RK2**: Good balance for moderate accuracy needs
4. **RK4**: Gold standard for high accuracy; most simulations use this
5. **Error Scaling**: Higher-order methods pay off for tight tolerances
6. **Stability**: Stiff systems need implicit methods or very small dt

## Practical Recommendations

**LIF/IF neurons**: RK4 with dt = τ/10  
**HH neurons**: RK4 or adaptive methods (scipy.integrate.odeint uses LSODA)  
**Networks**: Euler with small dt (0.01-0.1 ms) for computational efficiency  
**Stiff systems**: Use scipy's implicit solvers (BDF, Radau)

## Rule of Thumb
$$\Delta t \leq \frac{\tau_{\text{fastest}}}{10}$$

where τ_fastest is the smallest time constant in your system.

## Further Exploration

- Implement adaptive step size (error control)
- Test implicit methods for stiff systems
- Compare with scipy's odeint/solve_ivp
- Profile computational efficiency for large-scale networks